# CRAG Interactive Demo (Gradio) — Self-Contained
Pick a PopQA question and watch the corrective pipeline run, with the **full decision trace**:
retrieved docs -> grader verdict + reasoning -> corrective branch -> final answer + exact-match.

**This notebook is standalone** — it loads the model, builds the corpus, and launches the demo from scratch.


## Section 0 — Installs (run once, then RESTART runtime)

In [1]:
!pip -q install "langchain>=0.3,<0.4" "langchain-community>=0.3,<0.4" "langgraph>=0.2" \
  "langchain-huggingface" "faiss-cpu" "sentence-transformers" "datasets" "gradio"
!pip -q install bitsandbytes accelerate ddgs hf_transfer
print("Installs done. Runtime > Restart session, then run from Section 1.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 57.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 100.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.4/155.4 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 99.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2

## Section 1 — Setup, model, embeddings

In [2]:
import os, warnings, logging
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("bitsandbytes").setLevel(logging.ERROR)
from huggingface_hub import login
try:
    from google.colab import userdata; login(token=userdata.get("HF_TOKEN")); print("HF login OK")
except Exception: print("HF token not set (slower downloads)")

HF login OK


In [3]:
import json, re, time, random
import numpy as np, torch, requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline, ChatHuggingFace
from langchain_community.vectorstores import FAISS
from ddgs import DDGS

EMBED_MODEL="BAAI/bge-small-en-v1.5"; CHUNK_SIZE=256; CHUNK_OVERLAP=32; TOP_K=5

In [5]:
MODEL_ID="unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
tok=AutoTokenizer.from_pretrained(MODEL_ID)
model=AutoModelForCausalLM.from_pretrained(MODEL_ID, device_map={"":0})
gen_pipe=pipeline("text-generation", model=model, tokenizer=tok,
                  max_new_tokens=256, do_sample=False,
                  return_full_text=False, repetition_penalty=1.1)
llm=ChatHuggingFace(llm=HuggingFacePipeline(pipeline=gen_pipe))
embeddings=HuggingFaceEmbeddings(model_name=EMBED_MODEL, encode_kwargs={"normalize_embeddings":True})
print("device:", next(model.parameters()).device, "| VRAM:", round(torch.cuda.memory_allocated()/1e9,1),"GB")
print(llm.invoke("Reply with exactly: CRAG backend OK").content)

model.safetensors:   0%|          | 0.00/5.55G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'repetition_penalty', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


device: cuda:0 | VRAM: 5.7 GB


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


CRAG backend OK


## Section 2 — Data + corpus

In [6]:
ds=load_dataset("akariasai/PopQA"); data=ds["test"]
def parse_gold_answers(ex): return [a.strip() for a in json.loads(ex["possible_answers"]) if a and a.strip()]
def exact_match(pred, gold):
    p=pred.lower().strip(); return any(g.lower() in p for g in gold)
random.seed(123); demo_set=random.sample(list(range(len(data))), 60)
demo_set=[data[i] for i in demo_set]
print("demo questions:", len(demo_set))

README.md:   0%|          | 0.00/1.69k [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


test.tsv:   0%|          | 0.00/5.21M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/14267 [00:00<?, ? examples/s]

demo questions: 60


In [8]:
import warnings, logging
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)

In [9]:
# Build a small corpus from the demo questions\' subjects
session=requests.Session()
session.headers.update({"User-Agent":"CRAG-demo/1.0 (educational)"})
session.mount("https://", HTTPAdapter(max_retries=Retry(total=5,backoff_factor=1.5,
    status_forcelist=[429,500,502,503,504],respect_retry_after_header=True)))
def fetch_wiki(title):
    r=session.get("https://en.wikipedia.org/w/api.php",
        params={"action":"query","prop":"extracts","explaintext":1,"titles":title,
                "format":"json","redirects":1}, timeout=30)
    r.raise_for_status(); return next(iter(r.json()["query"]["pages"].values())).get("extract","")
splitter=RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    AutoTokenizer.from_pretrained(EMBED_MODEL), chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)

titles=sorted({e["s_wiki_title"] for e in demo_set})
print(f"Fetching {len(titles)} articles...")
docs=[]
for i,t in enumerate(titles,1):
    try:
        x=fetch_wiki(t)
        if x:
            for ch in splitter.split_text(x):
                docs.append(Document(page_content=ch, metadata={"title":t,"source":"wikipedia"}))
    except Exception: pass
    time.sleep(0.4)
    if i%20==0: print(f"  ...{i}")
retriever=FAISS.from_documents(docs, embeddings).as_retriever(search_kwargs={"k":TOP_K})
print(f"Corpus: {len(docs)} chunks. Retriever ready.")

Fetching 60 articles...
  ...20
  ...40
  ...60
Corpus: 863 chunks. Retriever ready.


## Section 3 — CRAG components (calibrated)

In [10]:
GRADER_PROMPT="""You are a retrieval evaluator. Decide whether the retrieved documents
are likely to help answer the question about the entity named in it.

Question:
{question}

Retrieved documents:
{documents}

Guidance:
- "Correct" if the documents are about the right entity AND plausibly contain the answer.
- "Ambiguous" if about the right entity but clearly lacking the specific fact.
- "Incorrect" ONLY if about a clearly DIFFERENT entity/topic. When unsure, prefer Correct/Ambiguous.

Respond with ONLY a JSON object:
{{"reasoning": "<one short sentence>", "score": "<Correct|Ambiguous|Incorrect>"}}"""
_VALID={"Correct","Ambiguous","Incorrect"}
def _extract_json(t):
    m=re.search(r"\{.*\}", t, re.DOTALL)
    if not m: return None
    try: return json.loads(m.group(0))
    except Exception: return None
def grade_retrieval(q, docs):
    dt="\n\n".join(f"[{i+1}] {d.page_content}" for i,d in enumerate(docs))
    resp=llm.invoke(GRADER_PROMPT.format(question=q, documents=dt))
    p=_extract_json(resp.content)
    if p and p.get("score") in _VALID: return {"score":p["score"],"reasoning":p.get("reasoning","")}
    low=resp.content.lower()
    for lab in ["incorrect","ambiguous","correct"]:
        if lab in low: return {"score":lab.capitalize(),"reasoning":"(parsed from text)"}
    return {"score":"Ambiguous","reasoning":"(unparseable; defaulted)"}

REWRITE_PROMPT="""Rewrite the question into a web search query. Keep all named entities,
add a type word if implied (song, film, book, person, city). Return ONLY the query.

Question: {question}
Search query:"""
def rewrite_query(q): return llm.invoke(REWRITE_PROMPT.format(question=q)).content.strip().strip('"')
def web_search(query, max_results=5):
    out=[]
    try:
        with DDGS() as d:
            for r in d.text(query, max_results=max_results):
                out.append(Document(page_content=f"{r.get('title','')}\n{r.get('body','')}",
                           metadata={"title":r.get('title',''),"source":"web","url":r.get('href','')}))
    except Exception as e: print("web_search error:",e)
    return out
GENERATOR_PROMPT="""Answer the question using ONLY the context below.
Be concise — give just the answer (a name, term, or short phrase).
If the context does not contain the answer, reply exactly: I don't know.

Context:
{context}

Question: {question}
Answer:"""
def generate_answer(q, docs):
    ctx="\n\n".join(f"[{i+1}] {d.page_content}" for i,d in enumerate(docs))
    return llm.invoke(GENERATOR_PROMPT.format(context=ctx, question=q)).content.strip()
def rerank_docs(q, docs, top_k=5):
    if not docs: return docs
    qv=np.array(embeddings.embed_query(q)); dv=np.array(embeddings.embed_documents([d.page_content for d in docs]))
    return [docs[i] for i in np.argsort(dv@qv)[::-1][:top_k]]
print("Components ready.")

Components ready.


## Section 4 — CRAG graph

In [11]:
from typing import TypedDict, List
from langgraph.graph import StateGraph, START, END
class CRAGState(TypedDict):
    question:str; documents:List[Document]; grade:str; generation:str; steps:List[str]
def node_retrieve(s): return {"documents":retriever.invoke(s["question"]), "steps":s.get("steps",[])+["retrieve"]}
def node_grade(s):
    r=grade_retrieval(s["question"], s["documents"]); return {"grade":r["score"], "steps":s["steps"]+[f"grade={r['score']}"]}
def node_websearch(s):
    web=web_search(rewrite_query(s["question"])); combined=s["documents"]+web
    docs=rerank_docs(s["question"], combined, top_k=TOP_K)
    lab="websearch(augment)" if s["grade"]=="Ambiguous" else "websearch(incorrect+augment)"
    return {"documents":docs, "steps":s["steps"]+[lab+"+rerank"]}
def node_generate(s): return {"generation":generate_answer(s["question"], s["documents"]), "steps":s["steps"]+["generate"]}
def route_after_grade(s): return "generate" if s["grade"]=="Correct" else "websearch"
def build_crag_app():
    b=StateGraph(CRAGState)
    b.add_node("retrieve",node_retrieve); b.add_node("grade",node_grade)
    b.add_node("websearch",node_websearch); b.add_node("generate",node_generate)
    b.add_edge(START,"retrieve"); b.add_edge("retrieve","grade")
    b.add_conditional_edges("grade", route_after_grade, {"generate":"generate","websearch":"websearch"})
    b.add_edge("websearch","generate"); b.add_edge("generate",END)
    return b.compile()
crag_app=build_crag_app(); print("Graph compiled.")

Graph compiled.


## Section 5 — Gradio demo

In [12]:
import gradio as gr

Q_CHOICES={}
for ex in demo_set:
    label=ex["question"]
    if label in Q_CHOICES: label=f"{ex['question']}  [{ex['id']}]"
    Q_CHOICES[label]=ex

def _fmt_docs(docs, limit=4):
    rows=[]
    for i,d in enumerate(docs[:limit],1):
        src=d.metadata.get("source","?"); title=d.metadata.get("title", d.metadata.get("url",""))
        snip=d.page_content.strip().replace("\n"," ")[:280]
        rows.append(f"**[{i}] ({src}: {title})**  {snip}...")
    extra=f"\n\n*(+{len(docs)-limit} more)*" if len(docs)>limit else ""
    return ("\n\n".join(rows)+extra) if rows else "_(none)_"

def _path(steps):
    pretty=[]
    for s in steps:
        if s=="retrieve": pretty.append("RETRIEVE")
        elif s.startswith("grade="): pretty.append(s.replace("grade=","GRADE: "))
        elif "websearch" in s: pretty.append("WEB SEARCH (correct)")
        elif s=="generate": pretty.append("GENERATE")
        else: pretty.append(s.upper())
    return "  →  ".join(pretty)

def run_demo(choice):
    ex=Q_CHOICES[choice]; q=ex["question"]; gold=parse_gold_answers(ex)
    retrieved=retriever.invoke(q)
    g=grade_retrieval(q, retrieved)
    out=crag_app.invoke({"question":q,"steps":[]})
    ans=out["generation"]; ok=exact_match(ans, gold)
    return (_path(out["steps"]),
            _fmt_docs(retrieved),
            f"**{g['score']}**\n\n_{g.get('reasoning','')}_",
            f"### Answer: {ans}\n\n**Gold:** {', '.join(gold)}\n\n"
            f"**Exact-match:** {'CORRECT' if ok else 'INCORRECT'}")

with gr.Blocks(title="CRAG Demo") as demo:
    gr.Markdown("# CRAG Pipeline Demo\nPick a PopQA question and trace the corrective pipeline.")
    with gr.Row():
        dd=gr.Dropdown(choices=list(Q_CHOICES.keys()), value=list(Q_CHOICES.keys())[0],
                       label="PopQA question", scale=4)
        btn=gr.Button("Run CRAG", variant="primary", scale=1)
    gr.Markdown("### Decision path"); path_box=gr.Markdown()
    with gr.Row():
        docs_box=gr.Markdown(label="Retrieved documents")
        grade_box=gr.Markdown(label="Grader verdict")
    answer_box=gr.Markdown()
    btn.click(run_demo, inputs=dd, outputs=[path_box, docs_box, grade_box, answer_box])

print("Demo built. Launch in the next cell.")

Demo built. Launch in the next cell.


## Section 6 — Launch
`share=True` gives a temporary public link (~72h). Re-run to relaunch.

In [13]:
demo.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1bd22979a01d20867b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
